# Stage 2 — Resistance Distance Spectral Stream

Implements reviewer-requested fixes for the resistance stream pipeline.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.linalg import eigh
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             roc_auc_score)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PROC_DIR = Path('processed')
FIG_DIR = Path('figures')
MODEL_DIR = Path('models')
FOLD_DIR = Path('folds')
for d in [PROC_DIR, FIG_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Omega_train = np.load(PROC_DIR / 'Omega_train.npy')
Omega_test = np.load(PROC_DIR / 'Omega_test.npy')
y = np.load(PROC_DIR / 'y_train.npy')
print('Loaded:', Omega_train.shape, Omega_test.shape, y.shape)


In [ ]:
def extract_mds_spectral_features(Omega: np.ndarray, k: int = 50) -> np.ndarray:
    """Extract top-k non-negative MDS eigenvalues as subject-level features."""
    n = Omega.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ (Omega ** 2) @ H
    eigvals, _ = eigh(B)
    eigvals_desc = np.maximum(eigvals[::-1], 0)
    top_k = eigvals_desc[:k]
    if top_k.shape[0] < k:
        top_k = np.pad(top_k, (0, k - top_k.shape[0]))
    return top_k.astype(np.float32)


def extract_batch_mds_features(Omega_batch: np.ndarray, k: int = 50) -> np.ndarray:
    features = np.zeros((Omega_batch.shape[0], k), dtype=np.float32)
    for i in tqdm(range(Omega_batch.shape[0]), desc='Extracting MDS features'):
        features[i] = extract_mds_spectral_features(Omega_batch[i], k=k)
    return features

K_MDS = 50
mds_features_train = extract_batch_mds_features(Omega_train, k=K_MDS)
mds_features_test = extract_batch_mds_features(Omega_test, k=K_MDS)
np.save(PROC_DIR / 'mds_features_train.npy', mds_features_train)
np.save(PROC_DIR / 'mds_features_test.npy', mds_features_test)
print('MDS feature shape:', mds_features_train.shape)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mds_features_train[:10].T, alpha=0.5)
axes[0].set_xlabel('Eigenvalue index')
axes[0].set_ylabel('Eigenvalue')
axes[0].set_title(f'Top-{K_MDS} MDS Eigenvalues (10 subjects)')

adhd_idx = np.where(y == 1)[0]
td_idx = np.where(y == 0)[0]
mean_adhd = mds_features_train[adhd_idx].mean(axis=0)
mean_td = mds_features_train[td_idx].mean(axis=0)
std_adhd = mds_features_train[adhd_idx].std(axis=0)
std_td = mds_features_train[td_idx].std(axis=0)

axes[1].plot(mean_adhd, label='ADHD', color='tomato', linewidth=2)
axes[1].fill_between(range(K_MDS), mean_adhd - std_adhd, mean_adhd + std_adhd, alpha=0.2, color='tomato')
axes[1].plot(mean_td, label='TD', color='steelblue', linewidth=2)
axes[1].fill_between(range(K_MDS), mean_td - std_td, mean_td + std_td, alpha=0.2, color='steelblue')
axes[1].set_xlabel('Eigenvalue index')
axes[1].set_ylabel('Eigenvalue')
axes[1].set_title('Mean MDS Spectrum by Group (mean ± std)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'mds_spectra.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
class ResistanceCNN(nn.Module):
    """3-layer CNN encoder for resistance matrices."""
    def __init__(self, feat_dim=128, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(128, feat_dim)
        self.classifier = nn.Linear(feat_dim, n_classes)

    def encode(self, x):
        x = self.features(x).flatten(1)
        return self.proj(x)

    def forward(self, x):
        emb = self.encode(x)
        logits = self.classifier(emb)
        return logits, emb


def train_cnn(Omega_train, y_train, Omega_val, y_val, epochs=30, batch_size=32, lr=1e-4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_train = torch.from_numpy(Omega_train).float().unsqueeze(1)
    X_val = torch.from_numpy(Omega_val).float().unsqueeze(1)
    y_train_t = torch.from_numpy(y_train).long()

    train_loader = DataLoader(TensorDataset(X_train, y_train_t), batch_size=batch_size, shuffle=True)

    model = ResistanceCNN(feat_dim=128, n_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_acc = -1.0
    best_model_state = None
    for _ in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits, _ = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits, _ = model(X_val.to(device))
            val_preds = torch.argmax(val_logits, dim=1).cpu().numpy()
            val_acc = accuracy_score(y_val, val_preds)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        _, train_features = model(X_train.to(device))
        _, val_features = model(X_val.to(device))
    return train_features.cpu().numpy(), val_features.cpu().numpy(), model, best_val_acc


In [ ]:
with open(FOLD_DIR / 'fold_0.json', 'r') as f:
    fold = json.load(f)
train_idx = fold['train_idx']
val_idx = fold['val_idx']

cnn_train_features, cnn_val_features, cnn_model, val_acc = train_cnn(
    Omega_train[train_idx], y[train_idx], Omega_train[val_idx], y[val_idx],
    epochs=30, batch_size=32, lr=1e-4
)
print('CNN validation accuracy:', round(val_acc, 4))

torch.save(cnn_model.state_dict(), MODEL_DIR / 'resistance_cnn_fold0.pth')


In [ ]:
def fuse_features(train_left, train_right, val_left, val_right, pca_components=50):
    train_concat = np.hstack([train_left, train_right])
    val_concat = np.hstack([val_left, val_right])
    pca = PCA(n_components=min(pca_components, train_concat.shape[1]))
    train_fused = pca.fit_transform(train_concat)
    val_fused = pca.transform(val_concat)
    return train_fused, val_fused, pca

mds_train_fold = mds_features_train[train_idx]
mds_val_fold = mds_features_train[val_idx]
fused_train, fused_val, pca = fuse_features(mds_train_fold, cnn_train_features, mds_val_fold, cnn_val_features, pca_components=50)
print('Fused training features:', fused_train.shape)
print('Fused validation features:', fused_val.shape)


In [ ]:
def evaluate_classifier(features_train, y_train, features_val, y_val):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(features_train)
    Xva = scaler.transform(features_val)
    clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED)
    clf.fit(Xtr, y_train)
    y_pred = clf.predict(Xva)
    y_proba = clf.predict_proba(Xva)[:, 1]
    return {
        'accuracy': accuracy_score(y_val, y_pred),
        'auc': roc_auc_score(y_val, y_proba),
        'f1': f1_score(y_val, y_pred),
        'y_pred': y_pred,
        'y_proba': y_proba,
    }

results = {}
results['MDS Only'] = evaluate_classifier(mds_train_fold, y[train_idx], mds_val_fold, y[val_idx])
results['CNN Only'] = evaluate_classifier(cnn_train_features, y[train_idx], cnn_val_features, y[val_idx])
results['Fused (MDS+CNN+PCA)'] = evaluate_classifier(fused_train, y[train_idx], fused_val, y[val_idx])

for name, met in results.items():
    print(f"{name:20s} | Acc={met['accuracy']:.4f} AUC={met['auc']:.4f} F1={met['f1']:.4f}")

results_df = pd.DataFrame([
    {'feature_set': k, 'accuracy': v['accuracy'], 'auc': v['auc'], 'f1': v['f1']}
    for k, v in results.items()
])
results_df.to_csv(PROC_DIR / 'resistance_stream_results.csv', index=False)
fused_res = results['Fused (MDS+CNN+PCA)']


In [ ]:
def run_cross_validation_fold(fold_idx, Omega, y, mds_features):
    with open(FOLD_DIR / f'fold_{fold_idx}.json', 'r') as f:
        fold = json.load(f)
    tr_idx, va_idx = fold['train_idx'], fold['val_idx']

    cnn_tr, cnn_va, _, _ = train_cnn(Omega[tr_idx], y[tr_idx], Omega[va_idx], y[va_idx], epochs=20, batch_size=32, lr=1e-4)
    fused_tr, fused_va, _ = fuse_features(mds_features[tr_idx], cnn_tr, mds_features[va_idx], cnn_va, pca_components=50)
    met = evaluate_classifier(fused_tr, y[tr_idx], fused_va, y[va_idx])
    return {'fold': fold_idx, 'accuracy': met['accuracy'], 'auc': met['auc'], 'f1': met['f1']}

cv_results = []
for fold_idx in range(5):
    r = run_cross_validation_fold(fold_idx, Omega_train, y, mds_features_train)
    cv_results.append(r)
    print(f"Fold {fold_idx}: Acc={r['accuracy']:.4f}, AUC={r['auc']:.4f}, F1={r['f1']:.4f}")

acc_mean, acc_std = np.mean([r['accuracy'] for r in cv_results]), np.std([r['accuracy'] for r in cv_results])
auc_mean, auc_std = np.mean([r['auc'] for r in cv_results]), np.std([r['auc'] for r in cv_results])
f1_mean, f1_std = np.mean([r['f1'] for r in cv_results]), np.std([r['f1'] for r in cv_results])

print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"AUC:      {auc_mean:.4f} ± {auc_std:.4f}")
print(f"F1:       {f1_mean:.4f} ± {f1_std:.4f}")

pd.DataFrame(cv_results).to_csv(PROC_DIR / 'resistance_stream_cv.csv', index=False)
pd.DataFrame({'metric':['accuracy','auc','f1'], 'mean':[acc_mean,auc_mean,f1_mean], 'std':[acc_std,auc_std,f1_std]}).to_csv(PROC_DIR / 'resistance_stream_summary.csv', index=False)


In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(y[val_idx], fused_res['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Fold 0)')

fpr, tpr, _ = roc_curve(y[val_idx], fused_res['y_proba'])
axes[1].plot(fpr, tpr, linewidth=2, label=f"AUC = {fused_res['auc']:.3f}")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'resistance_stream_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResistanceCNN(feat_dim=128, n_classes=2).to(device)
model.load_state_dict(torch.load(MODEL_DIR / 'resistance_cnn_fold0.pth', map_location=device))
model.eval()

X_train_tensor = torch.from_numpy(Omega_train).float().unsqueeze(1)
X_test_tensor = torch.from_numpy(Omega_test).float().unsqueeze(1)
with torch.no_grad():
    _, cnn_features_all_train = model(X_train_tensor.to(device))
    _, cnn_features_all_test = model(X_test_tensor.to(device))

all_concat_train = np.hstack([mds_features_train, cnn_features_all_train.cpu().numpy()])
all_concat_test = np.hstack([mds_features_test, cnn_features_all_test.cpu().numpy()])
all_pca = PCA(n_components=min(50, all_concat_train.shape[1]))
fused_train_all = all_pca.fit_transform(all_concat_train)
fused_test_all = all_pca.transform(all_concat_test)

np.save(PROC_DIR / 'resistance_stream_features_train.npy', fused_train_all)
np.save(PROC_DIR / 'resistance_stream_features_test.npy', fused_test_all)
print('Resistance stream features saved:', fused_train_all.shape, fused_test_all.shape)


### Stage 2 completion checklist

- ✓ `np.linalg.pinv(..., rcond=1e-6)` used in Stage 2 upstream resistance construction.
- ✓ MDS top-K eigenvalues treated as subject-level features.
- ✓ CNN justification included.
- ✓ Fusion uses PCA before concatenation blowup.
- ✓ Metrics reported as mean ± std.
